# CASE 01. Data Quality Check & Preprocessing

UCI Online Retail II의 두 연도 시트를 병합하고, 고객 분석과 매출 분석에 사용할 정제 데이터를 만드는 노트북입니다.

이 노트북은 다음 원칙을 사용합니다.

- 원본 데이터는 수정하지 않습니다.
- `Customer ID`가 없는 거래는 익명 매출 분석에는 쓸 수 있지만 고객 단위 분석(RFM, 재구매, 코호트)에는 쓸 수 없으므로 별도로 분리합니다.
- 음수 수량 및 취소 송장은 삭제 전에 플래그를 만들고 `returns_df`로 보존합니다.
- 최종 `cleaned_retail_data.csv`는 고객 식별이 가능하고 수량과 가격이 양수인 완료 구매만 포함합니다.
- 취소율, 반품률, 순매출을 분석할 때는 `returns_df` 또는 플래그가 포함된 `retail_df`를 사용해야 합니다.

## Goal

1. Excel의 모든 연도 시트를 하나의 DataFrame으로 병합합니다.
2. 결측치와 핵심 컬럼의 타입을 점검합니다.
3. 고객 ID 결측 거래와 취소·반품 거래를 분석 목적에 맞게 분리합니다.
4. 거래 금액 `Total_Revenue`를 계산합니다.
5. 저장 전 품질 검사를 통과한 데이터를 CSV로 내보냅니다.

## Data Scope and Limitations

이 데이터의 관측 기간은 2009-12-01부터 2011-12-09까지입니다. 따라서 이 노트북은 현재 영국 소매시장이나 2026년 소비 트렌드를 설명하기 위한 것이 아니라, 대규모 거래 데이터의 품질 관리와 고객·매출 분석 방법론을 검증하기 위한 포트폴리오 사례입니다.

해석 시 다음 원칙을 적용합니다.

- 결과는 2009~2011년 관측 기간 내부의 거래 패턴으로만 해석합니다.
- 데이터의 날짜를 최근 연도로 임의 변경하지 않습니다.
- 현재 시장에 대한 결론이나 전략 제안에는 최신 데이터로 재검증이 필요합니다.
- 이 케이스의 평가 대상은 데이터 정제, 지표 정의, 분석 재현성 및 논리적 해석입니다.

## Setup

노트북을 저장소 루트 또는 `notebooks` 폴더에서 실행해도 경로를 찾을 수 있도록 후보 경로를 확인합니다. 원본 Excel은 약 106만 행이므로 로딩에 시간이 걸릴 수 있습니다.

In [ ]:
from pathlib import Path

import pandas as pd

pd.set_option("display.max_columns", None)
pd.set_option("display.float_format", lambda value: f"{value:,.2f}")

CASE_NAME = "01_retail_customer_analytics"
case_dir_candidates = [
    Path.cwd(),
    Path.cwd().parent,
    Path.cwd() / "cases" / CASE_NAME,
]

case_dir = next(
    (path.resolve() for path in case_dir_candidates
     if (path / "data" / "raw" / "online_retail_II.xlsx").exists()),
    None,
)

if case_dir is None:
    raise FileNotFoundError(
        "online_retail_II.xlsx를 찾지 못했습니다. 저장소 루트 또는 notebooks 폴더에서 실행하세요."
    )

raw_path = case_dir / "data" / "raw" / "online_retail_II.xlsx"
processed_dir = case_dir / "data" / "processed"
output_path = processed_dir / "cleaned_retail_data.csv"

print(f"Input : {raw_path}")
print(f"Output: {output_path}")

## Steps

### 1. 모든 시트 읽기 및 병합

`sheet_name=None`으로 모든 시트를 읽습니다. 각 행의 출처를 추적할 수 있도록 `Source_Sheet`를 추가한 뒤 세로로 병합합니다.

In [ ]:
excel_file = pd.ExcelFile(raw_path, engine="openpyxl")
print("Sheets:", excel_file.sheet_names)

sheet_frames = pd.read_excel(
    excel_file,
    sheet_name=None,
)

expected_columns = {
    "Invoice", "StockCode", "Description", "Quantity",
    "InvoiceDate", "Price", "Customer ID", "Country",
}

for sheet_name, frame in sheet_frames.items():
    missing_columns = expected_columns.difference(frame.columns)
    if missing_columns:
        raise ValueError(f"{sheet_name}: 필수 컬럼 누락 {sorted(missing_columns)}")

retail_df = pd.concat(
    [frame.assign(Source_Sheet=sheet_name) for sheet_name, frame in sheet_frames.items()],
    ignore_index=True,
)

print(f"Merged shape: {retail_df.shape[0]:,} rows x {retail_df.shape[1]} columns")
retail_df.head()

### 2. 타입 정규화 및 기본 품질 프로파일

문자열 앞뒤 공백을 제거하고 날짜·수치 타입을 명시적으로 변환합니다. 고객 ID는 소수점 표기를 제거하되 결측치는 유지할 수 있도록 pandas의 nullable string 타입으로 저장합니다. 변환 전에 소수부가 있는 비정상 고객 ID가 없는지 검증합니다.

In [ ]:
for column in ["Invoice", "StockCode", "Description", "Country", "Source_Sheet"]:
    retail_df[column] = retail_df[column].astype("string").str.strip()

retail_df["InvoiceDate"] = pd.to_datetime(retail_df["InvoiceDate"], errors="coerce")
retail_df["Quantity"] = pd.to_numeric(retail_df["Quantity"], errors="coerce")
retail_df["Price"] = pd.to_numeric(retail_df["Price"], errors="coerce")

customer_id_numeric = pd.to_numeric(retail_df["Customer ID"], errors="coerce")
fractional_customer_id = (
    customer_id_numeric.notna()
    & customer_id_numeric.mod(1).ne(0)
)
if fractional_customer_id.any():
    raise ValueError("소수부가 있는 Customer ID가 발견되었습니다. 원본 값을 확인하세요.")

retail_df["Customer ID"] = customer_id_numeric.astype("Int64").astype("string")

profile = pd.DataFrame({
    "dtype": retail_df.dtypes.astype(str),
    "missing_count": retail_df.isna().sum(),
    "missing_rate_pct": retail_df.isna().mean().mul(100),
    "unique_count": retail_df.nunique(dropna=True),
}).sort_values("missing_rate_pct", ascending=False)

profile

In [ ]:
quality_overview = pd.Series({
    "rows": len(retail_df),
    "columns": retail_df.shape[1],
    "exact_duplicate_rows": retail_df.duplicated().sum(),
    "date_min": retail_df["InvoiceDate"].min(),
    "date_max": retail_df["InvoiceDate"].max(),
    "countries": retail_df["Country"].nunique(dropna=True),
})
quality_overview

### 3. `Customer ID` 결측치 확인 및 처리

고객 ID는 임의값으로 대체하지 않습니다. 하나의 대체 ID를 넣으면 서로 다른 익명 고객이 동일 인물로 합쳐지고, 평균 주문금액·재구매율·RFM 결과가 왜곡됩니다.

대신 두 데이터셋으로 분리합니다.

- `anonymous_transactions`: 고객 ID가 없어 상품·국가·전체 매출 집계에만 사용할 거래
- 고객 ID가 있는 거래: 이후 고객 단위 분석용 정제 후보

결측 고객 거래를 제외한 비율을 반드시 기록해 데이터 손실 규모를 확인합니다.

In [ ]:
missing_customer_mask = retail_df["Customer ID"].isna()
anonymous_transactions = retail_df.loc[missing_customer_mask].copy()
identified_transactions = retail_df.loc[~missing_customer_mask].copy()

customer_id_summary = pd.Series({
    "all_rows": len(retail_df),
    "missing_customer_rows": missing_customer_mask.sum(),
    "missing_customer_rate_pct": missing_customer_mask.mean() * 100,
    "identified_customer_rows": (~missing_customer_mask).sum(),
    "unique_identified_customers": retail_df["Customer ID"].nunique(dropna=True),
})
customer_id_summary

### 4. 취소·반품 거래 식별 및 분리

이 데이터에서는 음수 `Quantity`가 반품 또는 거래 취소를 나타낼 수 있고, `Invoice`가 `C`로 시작하는 행은 취소 송장입니다. 두 조건을 각각 플래그로 보존하고, 어느 하나라도 해당하면 취소·반품 데이터로 분류합니다.

두 플래그가 일치하지 않는 행은 자동 삭제하지 않고 점검 대상으로 남깁니다. 이는 데이터 오류일 수도 있고 조정 거래일 수도 있기 때문입니다.

In [ ]:
retail_df["Is_Cancelled_Invoice"] = (
    retail_df["Invoice"].str.upper().str.startswith("C", na=False)
)
retail_df["Is_Negative_Quantity"] = retail_df["Quantity"].lt(0)
retail_df["Is_Cancelled_or_Returned"] = (
    retail_df["Is_Cancelled_Invoice"] | retail_df["Is_Negative_Quantity"]
)

returns_df = retail_df.loc[retail_df["Is_Cancelled_or_Returned"]].copy()
completed_candidates = retail_df.loc[~retail_df["Is_Cancelled_or_Returned"]].copy()

cancellation_summary = pd.Series({
    "cancelled_invoice_rows": retail_df["Is_Cancelled_Invoice"].sum(),
    "negative_quantity_rows": retail_df["Is_Negative_Quantity"].sum(),
    "cancelled_or_returned_rows": retail_df["Is_Cancelled_or_Returned"].sum(),
    "flag_mismatch_rows": (
        retail_df["Is_Cancelled_Invoice"]
        != retail_df["Is_Negative_Quantity"]
    ).sum(),
})
cancellation_summary

In [ ]:
# 플래그 불일치 행은 별도로 검토합니다. 출력 행 수는 제한합니다.
flag_mismatches = retail_df.loc[
    retail_df["Is_Cancelled_Invoice"] != retail_df["Is_Negative_Quantity"],
    ["Invoice", "StockCode", "Quantity", "Price", "InvoiceDate", "Customer ID"],
]
flag_mismatches.head(20)

### 5. 거래 금액 파생 변수 생성

`Total_Revenue = Quantity × Price`로 계산합니다. 따라서 반품 행은 음수 금액이 되어 순매출 계산에 사용할 수 있습니다. 다만 최종 고객 구매 데이터에는 완료된 양수 거래만 남깁니다.

In [ ]:
retail_df["Total_Revenue"] = retail_df["Quantity"] * retail_df["Price"]
returns_df = retail_df.loc[retail_df["Is_Cancelled_or_Returned"]].copy()
completed_candidates = retail_df.loc[~retail_df["Is_Cancelled_or_Returned"]].copy()

revenue_check = retail_df[["Quantity", "Price", "Total_Revenue"]].describe(
    percentiles=[0.01, 0.25, 0.5, 0.75, 0.99]
)
revenue_check

### 6. 고객·매출 분석용 데이터 정제

최종 데이터는 아래 조건을 모두 만족하는 행으로 제한합니다.

- 고객 ID가 존재함
- 취소·반품 플래그가 아님
- `Quantity > 0`, `Price > 0`
- 송장 번호, 상품 코드, 거래 일시가 존재함

설명이 없는 상품은 상품 코드로 추적할 수 있으므로 자동 제외하지 않습니다. 정확히 동일한 행도 고유 라인 ID가 없어 정상 반복 구매와 구분하기 어렵기 때문에 현 단계에서는 제거하지 않고 건수만 기록합니다.

In [ ]:
required_value_mask = (
    retail_df["Invoice"].notna()
    & retail_df["StockCode"].notna()
    & retail_df["InvoiceDate"].notna()
    & retail_df["Customer ID"].notna()
    & retail_df["Quantity"].gt(0)
    & retail_df["Price"].gt(0)
    & ~retail_df["Is_Cancelled_or_Returned"]
)

cleaned_df = retail_df.loc[required_value_mask].copy()
cleaned_df = cleaned_df.sort_values(
    ["InvoiceDate", "Invoice", "StockCode"],
    kind="stable",
).reset_index(drop=True)

exclusion_summary = pd.Series({
    "source_rows": len(retail_df),
    "cleaned_rows": len(cleaned_df),
    "excluded_rows": len(retail_df) - len(cleaned_df),
    "retained_rate_pct": len(cleaned_df) / len(retail_df) * 100,
    "anonymous_rows": retail_df["Customer ID"].isna().sum(),
    "cancelled_or_returned_rows": retail_df["Is_Cancelled_or_Returned"].sum(),
    "non_positive_price_rows": retail_df["Price"].le(0).sum(),
    "exact_duplicates_in_cleaned": cleaned_df.duplicated().sum(),
})
exclusion_summary

## Checks

저장 전에 분석 계약을 검증합니다. 하나라도 실패하면 CSV를 저장하지 않고 원인을 먼저 확인합니다.

In [ ]:
assert len(retail_df) == sum(len(frame) for frame in sheet_frames.values())
assert cleaned_df["Customer ID"].notna().all()
assert cleaned_df["Quantity"].gt(0).all()
assert cleaned_df["Price"].gt(0).all()
assert cleaned_df["Total_Revenue"].gt(0).all()
assert ~cleaned_df["Is_Cancelled_or_Returned"].any()
assert cleaned_df["InvoiceDate"].notna().all()

print("All pre-save quality checks passed.")

### 7. 정제 데이터 저장

`utf-8-sig` 인코딩은 pandas뿐 아니라 Excel에서 CSV를 열 때도 한글과 특수 문자가 깨질 가능성을 줄입니다.

In [ ]:
processed_dir.mkdir(parents=True, exist_ok=True)
cleaned_df.to_csv(output_path, index=False, encoding="utf-8-sig")

print(f"Saved: {output_path}")
print(f"Rows : {len(cleaned_df):,}")
print(f"Size : {output_path.stat().st_size / (1024 ** 2):,.2f} MB")

## Next Steps

노트북을 실행한 뒤 다음 항목을 결과와 함께 기록합니다.

1. 시트별 행 수와 병합 후 총 행 수
2. `Customer ID` 결측 건수·비율 및 결측 거래의 매출 비중
3. 취소 송장과 음수 수량의 건수·불일치 사례
4. 가격 0 이하, 필수 키 결측, 중복 후보의 규모
5. 원본 대비 최종 데이터의 보존율

이후 CASE 01의 EDA에서는 `cleaned_df`로 고객·상품·국가별 양수 매출을 분석하고, 취소율·반품률·순매출은 플래그가 보존된 `retail_df`를 기준으로 별도 계산합니다.